# 04. CatBoost Baseline

This notebook establishes a reproducible CatBoost baseline on the minimally prepared dataset, evaluates threshold-dependent and probability-based metrics, and persists experiment artifacts.

In [1]:
from pathlib import Path
import sys

import pandas as pd

## Environment Setup

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"

EXPERIMENT_ID = "catboost_v0_raw_minimal_cv5"
RUN_TRAINING = True

Project root configured.
Processed directory configured.


In [3]:
from src.churn_ml.features import load_dataset
from src.churn_ml.modeling import (
    ExperimentConfig,
    load_experiment,
    save_experiment_result,
    summarize_target,
    build_experiment_summary,
)
from src.churn_ml.models.catboost import (
    CatBoostConfig,
    run_catboost_experiment,
)
from src.churn_ml.metrics import (
    optimize_threshold_by_folds,
)

c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Loading

In [4]:
dataset = load_dataset(
    version="v0_raw_minimal",
    processed_dir=PROCESSED_DIR,
)

print(f"Dataset version: {dataset.version}")
print(f"Train shape:     {dataset.X_train.shape}")
print(f"Target shape:    {dataset.y_train.shape}")
print(f"Test shape:      {dataset.X_test.shape}")

Dataset version: v0_raw_minimal
Train shape:     (10000, 205)
Target shape:    (10000,)
Test shape:      (2500, 205)


## Target Distribution

In [5]:
target_summary = summarize_target(dataset.y_train)

display(
    target_summary.style.format(
        {"proportion": "{:.2%}"}
    )
)

,class,count,proportion
0,0,8695,86.95%
1,1,1305,13.05%


## CatBoost Baseline

In [7]:
catboost_config = CatBoostConfig.default()
experiment_config = ExperimentConfig.default()

if RUN_TRAINING:

    catboost_experiment = run_catboost_experiment(
        dataset=dataset,
        config=experiment_config,
        model_config=catboost_config,
        experiment_id=EXPERIMENT_ID,
    )

    save_experiment_result(
        result=catboost_experiment.result,
        experiments_dir=EXPERIMENTS_DIR,
        fold_metrics=catboost_experiment.fold_metrics,
        oof_predictions=catboost_experiment.oof_predictions,
        test_predictions=catboost_experiment.test_predictions,
        overwrite=True,
    )
else:
    catboost_experiment = load_experiment(
        experiment_id=EXPERIMENT_ID,
        experiments_dir=EXPERIMENTS_DIR,
    )

Starting CatBoost cross-validation: 5 folds, 10,000 rows, 205 features


CatBoost CV: 100%|██████████| 5/5 [09:43<00:00, 116.69s/fold, balanced_accuracy=0.7872, best_iteration=611, roc_auc=0.9586]


## Experiment Results

In [8]:
display(catboost_experiment.fold_metrics)

display(
    pd.Series(
        catboost_experiment.result.metrics,
        name="value",
    ).to_frame()
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,decision_threshold,training_time_seconds,best_iteration
0,1,0.790897,0.970582,0.853891,0.149565,0.5,51.847443,609
1,2,0.770780,0.957099,0.809683,0.166769,0.5,157.620010,818
2,3,0.798078,0.960454,0.819719,0.164079,0.5,41.163835,473
3,4,0.822983,0.964896,0.841484,0.153321,0.5,274.152740,758
4,5,0.787159,0.958559,0.808492,0.166829,0.5,57.969307,611


,value
balanced_accuracy,0.793979
roc_auc,0.962040
average_precision,0.824716
log_loss,0.160113
decision_threshold,0.500000
optimized_threshold_oof,0.174000
optimized_balanced_accuracy_oof,0.894566
balanced_accuracy_mean,0.793979
balanced_accuracy_std,0.019054
roc_auc_mean,0.962318


In [9]:
summary = build_experiment_summary(
    catboost_experiment.result
)

display(
    summary.style.format(
        {"value": "{:.4f}"}
    )
)

,metric,value
0,Balanced Accuracy @ default threshold,0.7940
1,Optimized OOF Balanced Accuracy,0.8946
2,Optimized OOF threshold,0.1740
3,ROC-AUC,0.9620
4,Average Precision,0.8247
5,Log Loss,0.1601
6,"Training time, minutes",9.7239


In [10]:
fold_columns = [
    "fold",
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "log_loss",
    "best_iteration",
    "training_time_seconds",
]

display(
    catboost_experiment.fold_metrics[
        fold_columns
    ].style.format(
        {
            "balanced_accuracy": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "training_time_seconds": "{:.1f}",
        }
    )
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,best_iteration,training_time_seconds
0,1,0.7909,0.9706,0.8539,0.1496,609,51.8
1,2,0.7708,0.9571,0.8097,0.1668,818,157.6
2,3,0.7981,0.9605,0.8197,0.1641,473,41.2
3,4,0.8230,0.9649,0.8415,0.1533,758,274.2
4,5,0.7872,0.9586,0.8085,0.1668,611,58.0


## Threshold Stability

In [11]:
fold_thresholds = optimize_threshold_by_folds(
    fold_metrics=catboost_experiment.fold_metrics,
    oof_predictions=catboost_experiment.oof_predictions,
)

display(
    fold_thresholds.style.format(
        {
            "optimized_threshold": "{:.3f}",
            "optimized_balanced_accuracy": "{:.4f}",
        }
    )
)

print(
    "Threshold mean: "
    f"{fold_thresholds['optimized_threshold'].mean():.3f}"
)

print(
    "Threshold std:  "
    f"{fold_thresholds['optimized_threshold'].std(ddof=1):.3f}"
)

,fold,optimized_threshold,optimized_balanced_accuracy
0,1,0.128,0.9135
1,2,0.184,0.8814
2,3,0.192,0.9020
3,4,0.169,0.9016
4,5,0.117,0.8864


Threshold mean: 0.158
Threshold std:  0.034
